In [1]:
account_path = '/content/drive/MyDrive/'

# Mount & Import

In [2]:
# mount the drive
from google.colab import drive
drive.mount('/content/drive/')
from google.colab import auth
auth.authenticate_user()

# load packages - basics
!pip install pynrrd
import os
import csv
import numpy as np
import pandas as pd
import math
import time
import json
import nrrd

Mounted at /content/drive/


In [3]:
# load packages - stats
!pip install scikit-posthocs
import scipy
from scipy import io
from scipy.fftpack import rfft, irfft, fftfreq
import scipy.stats as stats
from scipy.stats import zscore

import statsmodels.api as sm
import statsmodels.formula.api as smf
import scikit_posthocs as sp
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import statsmodels

import random
from random import randrange, shuffle
import itertools
from itertools import combinations
import sklearn
from sklearn.decomposition import PCA, FastICA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, accuracy_score

In [4]:
# load packages - plotting and template settings
import matplotlib as mpl
from matplotlib import pyplot as plt
%matplotlib inline

from matplotlib.patches import Rectangle
import matplotlib.patches as patches
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib import animation
from mpl_toolkits.axes_grid1 import make_axes_locatable
import seaborn as sns

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [5]:
# load custom functions and sleap skeleton
%cd '/content/drive/MyDrive/Yu_escape_paper/Colab_github/src'
%run jy_utils.ipynb
sleap_nodes = ['le','re','fb','mb','rb','t1','t2','t3','tt'] # sleap_nodes_jy

/content/drive/MyDrive/Yu_escape_paper/Colab_github/src


# Experient list

In [6]:
# directories for the data and analysis
gsheet = 'https://docs.google.com/spreadsheets/d/1zJNHJJECC8FGB9S8tIh6QnoRihyoTZyKAgS-SE8ojwU/edit?gid=1421697431#gid=1421697431'
tab_name = 'loom_gsize'

experiment_class = 'Loom escape - group size'
path = account_path + 'Yu_escape_paper/Behavior/' + experiment_class
data_path = path + '/Data/'
save_path = path + '/Analysis/'
figure_path = path + '/Figures/'

metadata = read_metadata(gsheet,tab_name)
experiments = metadata.index

,experiment,fish_num,rig_mm,vid_px,rig_coord,loomX,loomZ,loom_dur
0,jy_240311_s16,1,265,945,"25,27;981,27;981,977;25,977",15,-30,6
1,jy_240311_s18,1,265,945,"25,27;981,27;981,977;25,977",15,-30,4
2,jy_240318_s20,1,265,927,"28,43;967,43;967,977;28,977",-15,30,5
3,jy_240318_s21,1,265,927,"28,43;967,43;967,977;28,977",-15,-30,3
4,jy_240318_s22,1,265,927,"28,43;967,43;967,977;28,977",-15,30,4
...,...,...,...,...,...,...,...,...
93,jy_240516_s10,8,265,904,"48,51;965,51;965,955;48,955",15,-30,3
94,jy_240516_s11,8,265,904,"48,51;965,51;965,955;48,955",15,30,3
95,jy_240516_s12,8,265,904,"48,51;965,51;965,955;48,955",-15,-30,3
96,jy_240516_s13,8,265,904,"48,51;965,51;965,955;48,955",-15,30,5


# process

# lev 0

In [ ]:
# extract tracking from h5 & tapping from csv
# save out lev0.npz

for exp in experiments[::]:
  load_path = data_path + exp + '/'
  output_path = save_path + exp + '/'; os.makedirs(output_path, exist_ok=True)

  # metadata
  fish_num = metadata.fish_num[exp]
  rig_mm = metadata.rig_mm[exp]; rig_coord = metadata.rig_coord[exp] # xy of four corners from top left, clockwise
  c1,c2,c3,c4 = rig_coord.split(';');
  x1,y1 = [int(c) for c in c1.split(',')]; x2,y2 = [int(c) for c in c2.split(',')]; x3,y3 = [int(c) for c in c3.split(',')]; x4,y4 = [int(c) for c in c4.split(',')]
  vid_px = (abs(x1-x2)+abs(y1-y4))/2; scale = rig_mm/vid_px # mm per pixel
  print('\nGroup:', exp, fish_num, 'fish')
  loomX = metadata.loomZ[exp]; loomY = metadata.loomX[exp]; loom_dur = metadata.loom_dur[exp]

  # -- step 1. Load h5 and csv -- #
  # timestamp.csv == absolute timestamps per video frame acquired
  # no header, one column
  time_sys = glob.glob(load_path + 'timestamp*.csv', recursive=True)[0]
  time_sys = np.array(pd.read_csv(time_sys, sep=',', header=0)).T[-1]
  date = str(time_sys[0][0:10]); start_h = int(time_sys[0][11:13])
  start_m = int(time_sys[0][14:16]); start_s = float(time_sys[0][17:24])
  start_time = start_h*3600+start_m*60+start_s # time of the first frame

  time_sec = [(float(i[11:13])*3600 + float(i[14:16])*60 + float(i[17:24])) - start_time for i in time_sys]
  time_sec = np.array(time_sec)
  fps=float(len(time_sec)/max(time_sec))
  print(f"behavior length = {max(time_sec):.2f} sec, fps = {fps:.2f}")

  # loom_time.csv == start of digital output triggers tapper
  # no header, 2 columns: True/False, timestamp
  looming = glob.glob(load_path + 'loom_timestamp*.csv', recursive=True)[0]
  loom = np.array(pd.read_csv(looming,sep=',', header=0, skiprows=1)).T
  loom_time = loom[1]
  loom_sec = [(float(i[11:13])*3600 + float(i[14:16])*60 + float(i[17:24])) - start_time for i in loom_time] # align to behavior time stamps
  loom_frames = find_nearest(time_sec,loom_sec) # align to behavior frames
  loom_onset = loom_frames[0]; loom_offset = loom_onset + int(loom_dur*fps)
  print(f"loom present at #{loom_onset} frame = {time_sec[loom_onset]:.2f} s for {loom_dur} sec")

  # load h5
  h5file = glob.glob(load_path + '*.h5', recursive=True)[0]
  h5file = h5py.File(h5file, 'r'); h5tracks = h5file['tracks'][:].T
  frame_num, _, _, track_num = h5tracks.shape
  print("frame_num x skeleton_size x 2 x fish_num", h5tracks.shape)

  # convert loom XY to same behavior tracking coordinates (px)
  # loomX [-13.25 to 13.25] = behaviorX (x1 to x2), loomY [10 to -10] = behaviorY (y1 to y4)
  start_x = loomX*((x2-x1)/26.5) + (x1+x2)/2; end_x = (x1+x2)/2
  start_y = loomY*((y4-y1)/20) + (y1+y4)/2; end_y = (y1+y4)/2
  xx = np.linspace(start_x,end_x,int(loom_dur*fps)); yy = np.linspace(start_y,end_y,int(loom_dur*fps));
  # pad with time series to full behavior length with invisible start/stop position
  loom_x = np.pad(xx, pad_width=((loom_frames[0], frame_num-loom_offset)), mode='constant', constant_values=((start_x,end_x)))
  loom_y = np.pad(yy, pad_width=((loom_frames[0], frame_num-loom_offset)), mode='constant', constant_values=((start_y,end_y)))

  # sanity check (1) consistent acquisition rate
  plt.figure(figsize=[4,1])
  plt.plot(time_sec); plt.fill_betweenx(plt.ylim(), loom_frames[0], loom_offset, color='red', alpha=0.25)
  plt.title('time should be linear + loom presence (red)',fontsize=10); sns.despine(); plt.show()

  # -- step 2. Extract tracking from h5 -- #
  group_data = process_h5(h5tracks,fish_num,sleap_nodes, exp)

  # organize them into group arrays and save as lev0_basics.npz
  f_bodylength_px, f_bodylength_mm, f_nosex, f_nosey, f_x, f_y, f_tailx, f_taily,\
  f_heading, f_tail_angle, f_speed, f_ang_speed \
  = group_arrays(group_data,fish_num,frame_num,fps,scale)

  # SAVING
  np.savez(output_path + 'lev0_loom.npz', loom_sec=loom_sec, loom_frames=loom_frames,\
            loom_len=int(loom_dur*fps),loom_x=loom_x,loom_y=loom_y)
  np.savez(output_path + 'lev0_basics.npz', fish_num=fish_num, fps=fps, frame_num=frame_num, \
            scale=scale, f_bodylength_px=f_bodylength_px, f_bodylength_mm=f_bodylength_mm,\
            f_nosex=f_nosex, f_nosey=f_nosey, f_x=f_x, f_y=f_y, f_tailx=f_tailx, f_taily=f_taily,\
            f_heading=f_heading, f_tail_angle=f_tail_angle, f_speed=f_speed, f_ang_speed=f_ang_speed)

  # sanity check (2) fish trajectory look clean and loom direction is not flipped
  fig, ax = plt.subplots(figsize=[3,3])
  ax.add_patch(patches.Rectangle((x1,y1),abs(y1-y4),abs(x1-x2),edgecolor=(0,0,0,0.25), facecolor='none', linewidth=1))
  for f in range(fish_num): plt.plot(f_x[f][::20],f_y[f][::20],linewidth=.5,alpha=.75)
  plt.scatter(loom_x[loom_onset:loom_offset:30],loom_y[loom_onset:loom_offset:30],color='gray',alpha=.5,s=43)
  plt.xlim(0,1000); plt.ylim(0,1000);
  for spine in ax.spines.values(): spine.set_visible(False)
  plt.show()

  # group level behaviors
  if fish_num == 1: continue
  ff_dist, ff_align, f_IID, f_IIA, f_closest_id, f_closest_dist, f_closest_align = analyze_neighbors(fish_num,f_heading,f_x,f_y,frame_num,scale,f_bodylength_px)
  group_area = 0 # default for 2-fish groups
  if fish_num > 2: group_area = fish_polygon_area2(f_x,f_y)

  # SAVING
  np.savez(output_path + 'lev0_group_basics.npz', group_area=group_area, ff_dist=ff_dist, \
          ff_align=ff_align, f_IID=f_IID, f_IIA=f_IIA, f_closest_id=f_closest_id,\
          f_closest_dist=f_closest_dist, f_closest_align=f_closest_align)

  # break

# lev 1 - fish to loom dist/align

In [ ]:
def shortest_distance_to_line(P1, P2, P):
    """Calculate the shortest distance from point P to the line segment defined by points P1 and P2."""
    # P = np.vstack((Mx, My)).T
    line_vec = P2 - P1; point_vec = P - P1
    line_len = np.linalg.norm(line_vec)
    line_unitvec = line_vec / line_len
    t = np.dot(point_vec, line_unitvec) / line_len
    t = max(0, min(1, t)); # Clamp t to the range [0, 1]

    nearest = P1 + t * line_vec
    distance = np.linalg.norm(nearest - P)
    return distance

In [ ]:
# group arrays of dist & alignment to loom

for exp in experiments[::]:
  load_path = save_path + exp + '/'
  output_path = save_path + exp + '/'; os.makedirs(output_path, exist_ok=True)

  # load data
  data = np.load(load_path + 'lev0_basics.npz', allow_pickle=True)
  fish_num, frame_num, scale = data['fish_num'], data['frame_num'], data['scale']
  f_x, f_y, f_heading = data['f_x'], data['f_y'], data['f_heading']
  data = np.load(load_path + 'lev0_loom.npz', allow_pickle=True)
  loom_sec, loom_frames, loom_len = data['loom_sec'], data['loom_frames'], data['loom_len']
  loom_on = loom_frames[0]; loom_off = loom_on + loom_len
  loom_x, loom_y = data['loom_x'], data['loom_y']
  print('\nGroup:', exp, fish_num, 'fish')

  # calculation - only during loom
  f_dist2loom = np.zeros((fish_num,loom_len)); f_align2loom = np.zeros((fish_num,loom_len))
  f_dist2traj = np.zeros((fish_num,loom_len)); P1 = np.array([loom_x[loom_on],loom_y[loom_on]]); P2 = np.array([loom_x[loom_off],loom_y[loom_off]])

  for f in range(fish_num):
    f_dist2loom[f] = np.sqrt((f_x[f,loom_on:loom_off]-loom_x[loom_on:loom_off])**2 + (f_y[f,loom_on:loom_off]-loom_y[loom_on:loom_off])**2)
    ang = calculate_angle(f_x[f,loom_on:loom_off],f_y[f,loom_on:loom_off],loom_x[loom_on:loom_off],loom_y[loom_on:loom_off])
    f_align2loom[f] = (((f_heading[f,loom_on:loom_off] - ang) + 180)  % 360 -180)

    moving_points = np.vstack((f_x[f,loom_on:loom_off], f_y[f,loom_on:loom_off])).T
    f_dist2traj[f] = np.array([shortest_distance_to_line(P1, P2, P) for P in moving_points])

  # make time_sec relative to tap for plotting
  rel_time = np.linspace(0,loom_len/fps,loom_len); rel_sec = np.round(rel_time,2)

  # visualize
  plt.figure(figsize=[4,2])
  for f in range(fish_num): plt.plot(rel_time,f_dist2loom[f]*scale,alpha=.7)
  plt.ylabel('distance (mm)'); sns.despine(); plt.show()
  plt.figure(figsize=[4,2])
  for f in range(fish_num): plt.plot(rel_time,f_dist2traj[f]*scale,alpha=.7)
  plt.ylabel('distance (mm)'); sns.despine();
  plt.figure(figsize=[4,2])
  for f in range(fish_num): plt.scatter(rel_time,f_align2loom[f]*scale,alpha=.6,s=2)
  plt.xlabel('time to offset (s)'); plt.ylabel('alignment (deg)'); plt.ylim(-180,180); sns.despine(); plt.show()

  # SAVING
  np.savez(output_path + 'lev1_fish2loom.npz', f_dist2loom=f_dist2loom, f_align2loom=f_align2loom, f_dist2traj=f_dist2traj)

  # break

# lev 2 - loom on fish

In [ ]:
# assumed sphere diameter ~43px (measurement from when sphere is on the screen)
# math: draw Isosceles triangle height= f_dist2loom, bottom edge=43 (21.5 each side)
# get the angle 21.5px edge is facing x2 = retinal occupancy

for exp in experiments[::]:
  load_path = save_path + exp + '/'; output_path = save_path + exp + '/';
  fish_num = metadata.fish_num[exp]; os.makedirs(output_path, exist_ok=True)

  # load data
  data = np.load(load_path + 'lev0_basics.npz', allow_pickle=True)
  fps = data['fps']
  data = np.load(load_path + 'lev0_loom.npz', allow_pickle=True)
  loom_sec, loom_frames, loom_len = data['loom_sec'], data['loom_frames'], data['loom_len']
  loom_on = loom_frames[0]; loom_off = loom_on + loom_len
  data  = np.load(load_path + 'lev1_fish2loom.npz', allow_pickle=True)
  f_dist2loom, f_align2loom = data['f_dist2loom'], data['f_align2loom']

  print('\nGroup:', exp, fish_num, 'fish')

  # calculation
  loom_occup_angles = np.zeros((fish_num,2,loom_len)); loom_subtend_angle = np.zeros((fish_num,loom_len))
  for f in range(fish_num):
    dist = f_dist2loom[f]; align = f_align2loom[f]
    # dist**2 + 21.5**2 = (dist to sphere edge)**2
    edge = np.sqrt(dist**2 + 21.5**2)
    # edge / np.sin(np.deg2rad(90)) = 21.5 / np.sin(np.deg2rad(half of the subtended angle))
    half_rad = np.arcsin((21.5 * np.sin(np.deg2rad(90))) / edge); half_ang = np.rad2deg(half_rad)
    loom_subtend_angle[f] = half_ang*2

    # get the angles for edges of sphere on egocentric ray casting plane
    min = align + half_ang; max = align - half_ang
    min[min < -180] = min[min < -180] + 360; max[max > 180] = max[max > 180] - 360
    loom_occup_angles[f] = np.array([min,max])

  # make time_sec relative to tap for plotting
  rel_time = np.linspace(0,loom_len/fps,loom_len); rel_sec = np.round(rel_time,2)

  # visualize
  plt.figure(figsize=[4,1])
  for f in range(fish_num): plt.plot(rel_time,loom_subtend_angle[f],alpha=.7)
  plt.ylabel('subtended'); plt.xlabel('time of loom (s)')
  sns.despine(); plt.show()

  # SAVING
  np.savez(output_path + 'lev2_loom2fish.npz', loom_occup_angles=loom_occup_angles, loom_subtend_angle=loom_subtend_angle)

  # break